# 가까운 지하철역 + 직선거리 변수 값 저장

(1) 임포트 및 API 키 설정

In [1]:
from dotenv import load_dotenv
import os
import pandas as pd
import requests
import time

# .env 파일에서 카카오 API 키 불러오기
load_dotenv()
KAKAO_API_KEY = os.environ.get('KAKAO_API_KEY')

(2) 데이터 로드

In [2]:
df = pd.read_csv('new_city.csv', encoding='utf-8-sig')
print("데이터 크기:", df.shape)
df.head()

데이터 크기: (90761, 34)


,도시명,시군구,번지,본번,부번,단지명,전용면적(㎡),계약년월,계약일,거래금액(만원),...,가장 가까운 IC와의 거리,계약연도,발표후경과년수,CPI,서울도심거리,단지별_세대수,도시별_세대수,기차역까지의거리,기차역이름,지하철호선개수
0,청라,인천광역시 서구 청라동,130-2,130,2,청라웰카운티1차,84.4963,201101,11,"30,200",...,2.795,2011,8,89.850,28.412513,692.0,27992.0,8.0317,인천역,1
1,청라,인천광역시 서구 청라동,130-2,130,2,청라웰카운티1차,84.7379,201012,20,"30,200",...,2.795,2010,7,86.373,28.412513,692.0,27992.0,8.0317,인천역,1
2,청라,인천광역시 서구 청라동,130-2,130,2,청라웰카운티1차,84.4963,201012,11,"28,500",...,2.795,2010,7,86.373,28.412513,692.0,27992.0,8.0317,인천역,1
3,청라,인천광역시 서구 청라동,130-2,130,2,청라웰카운티1차,84.4963,201011,18,"30,600",...,2.795,2010,7,86.373,28.412513,692.0,27992.0,8.0317,인천역,1
4,청라,인천광역시 서구 청라동,130-2,130,2,청라웰카운티1차,84.4963,201011,16,"27,476",...,2.795,2010,7,86.373,28.412513,692.0,27992.0,8.0317,인천역,1


(3) 주소 -> 좌표 변환 함수

In [3]:
# 주소 문자열을 입력받아 위도, 경도 반환
def get_coordinates(address):
    url = "https://dapi.kakao.com/v2/local/search/keyword.json"
    headers = {'Authorization': f'KakaoAK {KAKAO_API_KEY}'}
    params = {'query': address}
    response = requests.get(url, headers=headers, params=params)
    result = response.json()
    if result['documents']:
        lat = float(result['documents'][0]['y'])
        lng = float(result['documents'][0]['x'])
        return lat, lng
    else:
        return None, None

(4) 주변 지하철역 탐색 함수

In [4]:
# 좌표 기준으로 반경 내 가장 가까운 지하철역 이름과 거리 반환
def search_nearest_station(lat, lng, radius=5000):
    url = "https://dapi.kakao.com/v2/local/search/keyword.json"
    headers = {'Authorization': f'KakaoAK {KAKAO_API_KEY}'}
    params = {
        'query':  '지하철역',
        'x':      lng,
        'y':      lat,
        'radius': radius,
        'sort':   'distance',
        'size':   1
    }
    response = requests.get(url, headers=headers, params=params)
    result = response.json()
    if result['documents']:
        place = result['documents'][0]
        return {
            'name':     place['place_name'],
            'distance': int(place['distance'])
        }
    else:
        return None

(5) 단지별 좌표 추출

In [5]:
# 중복 제거한 단지별로 좌표 API 호출
unique_complexes = df[['단지명', '시군구', '도로명']].drop_duplicates().reset_index(drop=True)
print(f"고유 단지 수: {len(unique_complexes)}")

coord_dict = {}

for _, row in unique_complexes.iterrows():
    apt_name = row['단지명']
    
    # 1차: 도로명
    full_address = f"{row['시군구']} {row['도로명']}"
    lat, lng = get_coordinates(full_address)

    # 2차: 단지명
    if lat is None:
        full_address = f"{row['시군구']} {apt_name}"
        lat, lng = get_coordinates(full_address)

    # 3차: 괄호 제거 후 단지명
    if lat is None:
        clean_name = apt_name.split('(')[0].strip()
        full_address = f"{row['시군구']} {clean_name}"
        lat, lng = get_coordinates(full_address)

    coord_dict[apt_name] = (lat, lng)
    print(f"  {apt_name}: ({lat}, {lng})")
    time.sleep(0.1)

고유 단지 수: 215
  청라웰카운티1차: (37.53915215032671, 126.6579507991236)
  힐데스하임: (37.528482962173534, 126.65162000586596)
  호반베르디움앤영무예다음: (37.5374052810806, 126.657385130548)
  호반베르디움(116-6): (37.5372412737735, 126.653055810092)
  청라하우스토리커낼뷰: (37.53492771899579, 126.64165844218626)
  서해그랑블: (37.53103297214876, 126.65629649817643)
  청라엘에이치: (37.52848927529058, 126.64676887354958)
  청라우미린: (37.52885888492079, 126.62719519355979)
  청라메이루즈 커낼파크뷰: (37.535232258095604, 126.65343282993709)
  청라자이: (37.528460211394005, 126.65895106788645)
  청라지구중흥S-CLASS2단지: (37.53494969407704, 126.65570828855589)
  청라제일풍경채: (37.53976098795881, 126.64685148525466)
  청라SKVIEW: (37.5257533008036, 126.629821639793)
  청라힐스테이트커낼뷰: (37.534910435002395, 126.64368828777843)
  청라반도유보라: (37.52518430936221, 126.62766825054952)
  청라29블럭호반베르디움: (37.527521215340926, 126.63859975379631)
  청라한화꿈에그린: (37.53780174480017, 126.63865197483484)
  청라한라비발디: (37.54032215960669, 126.63846556085076)
  청라힐스테이트: (37.52598685490953, 126.6284493716

(6) 가장 가까운 지하철역 탐색

In [6]:
# 단지별 좌표로 가장 가까운 지하철역을 탐색하고 결과를 딕셔너리 저장
station_name_dict = {}
station_dist_dict = {}

for apt_name, (lat, lng) in coord_dict.items():
    if lat is None:
        station_name_dict[apt_name] = None
        station_dist_dict[apt_name] = None
        continue

    station = search_nearest_station(lat, lng)
    time.sleep(0.1)

    if station is not None:
        station_name_dict[apt_name] = station['name']
        station_dist_dict[apt_name] = station['distance']
    else:
        station_name_dict[apt_name] = None
        station_dist_dict[apt_name] = None

    print(f"  {apt_name} → {station_name_dict[apt_name]} {station_dist_dict[apt_name]}m")

  청라웰카운티1차 → 서구청역 인천2호선 1755m
  힐데스하임 → 가정역 인천2호선 2137m
  호반베르디움앤영무예다음 → 서구청역 인천2호선 1871m
  호반베르디움(116-6) → 서구청역 인천2호선 2234m
  청라하우스토리커낼뷰 → 청라국제도시역 공항철도 1번출구 2634m
  서해그랑블 → 가정역 인천2호선 1820m
  청라엘에이치 → 가정역 인천2호선 2559m
  청라우미린 → 청라국제도시역 공항철도 1번출구 2901m
  청라메이루즈 커낼파크뷰 → 가정역 인천2호선 6번출구 2161m
  청라자이 → 가정역 인천2호선 1505m
  청라지구중흥S-CLASS2단지 → 가정역 인천2호선 2069m
  청라제일풍경채 → 청라국제도시역 공항철도 1번출구 2519m
  청라SKVIEW → 청라국제도시역 공항철도 1번출구 3263m
  청라힐스테이트커낼뷰 → 청라국제도시역 공항철도 1번출구 2736m
  청라반도유보라 → 청라국제도시역 공항철도 1번출구 3310m
  청라29블럭호반베르디움 → 가정역 인천2호선 6번출구 3166m
  청라한화꿈에그린 → 청라국제도시역 공항철도 1번출구 2223m
  청라한라비발디 → 청라국제도시역 공항철도 1번출구 1979m
  청라힐스테이트 → 청라국제도시역 공항철도 1번출구 3226m
  호반베르디움 → 가정역 인천2호선 1732m
  청라동양엔파트 → 청라국제도시역 공항철도 1번출구 3375m
  한일베라체 → 가정역 인천2호선 6번출구 2958m
  청라 엑슬루타워 → 가정역 인천2호선 2189m
  NPART → 가정역 인천2호선 2881m
  청라지구중흥S-CLASS1단지 → 서구청역 인천2호선 2279m
  청라롯데캐슬 → 가정역 인천2호선 2462m
  청라반도유보라2.0 → 청라국제도시역 공항철도 1번출구 3425m
  청라린스트라우스 → 가정역 인천2호선 2458m
  청라한양수자인 → 청라국제도시역 공항철도 1번출구 3062m
  청라더샵레이크파크 → 청라국제도시역 공항철도 1번출구 2897

(7) 컬럼 추가 및 csv 저장

In [7]:
# 단지명 기준으로 컬럼 추가 후 CSV 저장 (거리 단위 m -> km로 변환)
df['가장 가까운 지하철역']           = df['단지명'].apply(lambda x: station_name_dict.get(x))
df['가장 가까운 지하철역까지의 거리'] = df['단지명'].apply(
    lambda x: round(station_dist_dict.get(x) / 1000, 3) if station_dist_dict.get(x) is not None else None)

print(df[['단지명', '가장 가까운 지하철역', '가장 가까운 지하철역까지의 거리']].drop_duplicates('단지명').to_string())

df.to_csv('new_city.csv', index=False, encoding='utf-8-sig')
print("최종 데이터 크기:", df.shape)

                        단지명           가장 가까운 지하철역  가장 가까운 지하철역까지의 거리
0                  청라웰카운티1차            서구청역 인천2호선              1.755
14                    힐데스하임             가정역 인천2호선              2.137
15             호반베르디움앤영무예다음            서구청역 인천2호선              1.871
16            호반베르디움(116-6)            서구청역 인천2호선              2.234
17               청라하우스토리커낼뷰     청라국제도시역 공항철도 1번출구              2.634
18                    서해그랑블             가정역 인천2호선              1.820
20                   청라엘에이치             가정역 인천2호선              2.559
21                    청라우미린     청라국제도시역 공항철도 1번출구              2.901
40             청라메이루즈 커낼파크뷰        가정역 인천2호선 6번출구              2.161
44                     청라자이             가정역 인천2호선              1.505
53         청라지구중흥S-CLASS2단지             가정역 인천2호선              2.069
128                 청라제일풍경채     청라국제도시역 공항철도 1번출구              2.519
129                청라SKVIEW     청라국제도시역 공항철도 1번출구              3.263
130              청라힐스테이트커낼뷰     청라